# Day 4: Files, JSON, Error Handling & Modules (28 Oct 2026)

**Week 1, Phase 1 (Foundation).** Day 4 of 7.

**Goal:** make programs remember things between runs (files and JSON), survive bad input without crashing (exceptions), and split code into reusable files (modules).

**Deliverables today:** `day04/contact_store.py` + `day04/main.py` (CRM import into a safe JSON store) and `day04/dsa.py` (top words, balanced brackets).

The full lesson, task brief and answers are in the master Google Doc. This notebook holds the runnable examples. Run the cells top to bottom; they create a `scratch/` folder next to the notebook, which you can delete afterwards.

## Recap answers (fill in before running anything)

1. What does `grid = [[0] * 2] * 2; grid[0][0] = 1; print(grid)` print, and why?
2. What does `by_city.setdefault(city, []).append(name)` do the first and second time a city appears?
3. What does `{n % 3 for n in range(10)}` produce, and why so few items?

*Your answers:*



## Concept 1: Files and paths

`open(path, mode, encoding="utf-8")` gives you a **file object**. The `with` block closes it for you, even when an error happens (a `try/finally` written for you). `pathlib.Path` builds paths that work on every OS, like `path.join()` in Node.

Modes: `"r"` read (default), `"w"` write (**empties the file first**), `"a"` append, `"x"` create only.

In [1]:
# Write a small text file, then read it back.
# pathlib: the modern way to handle paths (like Node's "path" module)
from pathlib import Path

# "/" joins path parts correctly on Windows, Mac and Linux
notes = Path("scratch") / "notes.txt"
notes.parent.mkdir(exist_ok=True)  # create the folder; no error if it exists

# "w" = write mode: creates the file, or EMPTIES it first if it already exists
# "with" closes the file automatically, even if an error happens inside
with open(notes, "w", encoding="utf-8") as f:
    # GOTCHA: write() does not add the newline for you
    f.write("T-101: login fails\n")
    f.write("T-102: export slow\n")

# "a" = append mode: keeps what is there and adds to the end
with open(notes, "a", encoding="utf-8") as f:
    f.write("T-103: wrong locale\n")

with open(notes, encoding="utf-8") as f:  # no mode given means "r" (read)
    # the whole file as ONE string, like fs.readFileSync(p, "utf8") in Node
    text = f.read()

print(len(text.splitlines()))  # -> 3
print(repr(text[:19]))  # -> 'T-101: login fails\n'  (repr shows the \n)
# REMEMBER: always open files with "with" and encoding="utf-8".

3
'T-101: login fails\n'


In [2]:
# Read a file line by line, and the pathlib shortcuts.
from pathlib import Path

notes = Path("scratch") / "notes.txt"

with open(notes, encoding="utf-8") as f:
    # a file object is iterable: one line per loop, so memory stays low
    for line in f:
        # strip() removes the "\n"; split(": ", 1) splits once, into 2 parts
        ticket_id, summary = line.strip().split(": ", 1)
        print(ticket_id, "|", summary)
# -> T-101 | login fails
# -> T-102 | export slow
# -> T-103 | wrong locale

# Shortcuts for small files: open, read/write and close in one call
report = Path("scratch") / "report.txt"
# write_text overwrites the file, like mode "w"
report.write_text("3 open tickets\n", encoding="utf-8")
print(report.read_text(encoding="utf-8").strip())  # -> 3 open tickets

print(report.exists(), report.name, report.suffix)  # -> True report.txt .txt
print(Path("scratch/missing.txt").exists())  # -> False

T-101 | login fails
T-102 | export slow
T-103 | wrong locale
3 open tickets
True report.txt .txt
False


**Remember:** always `with open(path, mode, encoding="utf-8")`. Mode `"w"` wipes the file the moment it opens. Paths are relative to where you *run* the command, unless you build them from `Path(__file__).parent`.

## Concept 2: JSON, the language every API speaks

JSON is **text**. `loads`/`dumps` work on strings (the "s" means string), `load`/`dump` work on open files. Like `JSON.parse()` / `JSON.stringify()` in JavaScript.

| JSON | Python |
|---|---|
| object `{}` | dict (keys always str) |
| array `[]` | list (tuples come back as lists) |
| `true` / `false` / `null` | `True` / `False` / `None` |
| (none) | set, datetime: `dumps` raises TypeError |

In [3]:
# JSON text <-> Python objects, in memory.
import json  # part of the standard library: nothing to install

# A CMS entry as it might arrive from an API: JSON is just TEXT
raw = '{"uid": "blt01", "tags": ["sales"], "live": true, "seo": null}'

entry = json.loads(raw)  # "load string": text -> Python, like JSON.parse()
print(type(entry))  # -> <class 'dict'>
print(entry["live"], entry["seo"])  # -> True None  (from true and null)

entry["tags"].append("2026")  # now it is a normal dict you can change
text = json.dumps(entry)  # "dump string": Python -> text, like JSON.stringify
print(text)
# -> {"uid": "blt01", "tags": ["sales", "2026"], "live": true, "seo": null}

# FDE TIP: indent=2 makes JSON readable for humans (logs, config files, diffs)
print(json.dumps({"id": 7, "ok": True}, indent=2))
# -> {
# ->   "id": 7,
# ->   "ok": true
# -> }

<class 'dict'>
True None
{"uid": "blt01", "tags": ["sales", "2026"], "live": true, "seo": null}
{
  "id": 7,
  "ok": true
}


In [4]:
# Round-trip surprises: JSON has fewer types than Python.
import json

record = {1: "one", "point": (3, 4), "note": None}
back = json.loads(json.dumps(record))  # Python -> JSON text -> Python
print(back)  # -> {'1': 'one', 'point': [3, 4], 'note': None}
# GOTCHA: int key 1 came back as the string "1", and the tuple as a list.
# JSON object keys are always strings, and JSON has arrays but no tuples.

try:
    json.dumps({"tags": {"sales", "vip"}})  # a set has no JSON equivalent
except TypeError as err:
    print("TypeError:", err)
    # -> TypeError: Object of type set is not JSON serializable

# Fix: convert the set to a sorted list first
print(json.dumps({"tags": sorted({"vip", "sales"})}))
# -> {"tags": ["sales", "vip"]}
# REMEMBER: loads/dumps work on strings; load/dump (no "s") work on files.

{'1': 'one', 'point': [3, 4], 'note': None}
TypeError: Object of type set is not JSON serializable
{"tags": ["sales", "vip"]}


In [5]:
# Save and load JSON files (the pattern you will use every day).
import json
from pathlib import Path

entries = [
    {"uid": "blt01", "title": "Pricing", "locale": "en-in"},
    {"uid": "blt02", "title": "Café menu", "locale": "fr-fr"},  # note the é
]
path = Path("scratch") / "entries.json"

with open(path, "w", encoding="utf-8") as f:
    # dump (no "s") writes straight into the open file
    # ensure_ascii=False writes "é" as it is, instead of an escape code
    json.dump(entries, f, indent=2, ensure_ascii=False)

with open(path, encoding="utf-8") as f:
    loaded = json.load(f)  # load (no "s") reads from the open file

print(loaded == entries)  # -> True  (same data came back)
print(loaded[1]["title"])  # -> Café menu
# the file itself is plain text you can search
print(path.read_text(encoding="utf-8").count('"uid"'))  # -> 2

True
Café menu
2


**Remember:** an API body is a string, so `json.loads`; a file is a file, so `json.load`. Int keys become strings and tuples become lists after a round trip. Use `indent=2, ensure_ascii=False` for files people will read.

## Concept 3: Exceptions and error handling

When something goes wrong Python **raises an exception**. Uncaught, it stops the program and prints a **traceback**: read it from the bottom up. The next cell fails on purpose so you can see one.

In [6]:
# An unhandled exception stops the program and prints a traceback.
contact = {"name": "Asha", "email": "asha@acme.in"}

def city_of(c):
    return c["city"]  # no "city" key, so this line raises KeyError

print("before")
print(city_of(contact))
print("after")  # never runs: the exception stopped the program

before


KeyError: 'city'

`try` / `except` is JavaScript's `try/catch`, plus `else` (runs only when no error) and `finally` (always runs). Catch the **specific** error you expect.

In [7]:
# try / except / else / finally, catching only the errors you expect.
raw_ages = ["34", "abc", "", "41"]  # messy data, like a CSV export from a CRM

ages = []
for raw in raw_ages:
    try:
        age = int(raw)  # the one line that can fail goes inside try
    except ValueError:  # catch the SPECIFIC error, never a bare "except:"
        # !r shows the quotes, so an empty string is visible
        print(f"skipping bad value {raw!r}")  # -> skipping bad value 'abc'
    else:
        ages.append(age)  # runs only if try did NOT raise
    finally:
        pass  # always runs (error or not): used for clean-up, like JS finally

# (the loop also printed: skipping bad value '')
print(ages)  # -> [34, 41]

skipping bad value 'abc'
skipping bad value ''
[34, 41]


In [8]:
# Load a JSON config safely: each failure gets its own clear message.
import json
from pathlib import Path

def load_settings(path: Path) -> dict:
    """Return settings from a JSON file, or {} if there is no file yet."""
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {}  # a first run is normal, not an error
    except json.JSONDecodeError as err:  # file exists but is not valid JSON
        # "raise ... from err" keeps the original cause in the traceback
        where = f"line {err.lineno}, column {err.colno}"
        msg = f"{path} is not valid JSON ({where})"
        raise ValueError(msg) from err

broken = Path("scratch") / "broken.json"
# a trailing comma makes this invalid JSON
broken.write_text('{"theme": "dark",}', encoding="utf-8")

print(load_settings(Path("scratch") / "nope.json"))  # -> {}
try:
    load_settings(broken)
except ValueError as err:
    print(err)  # -> scratch/broken.json is not valid JSON (line 1, column 17)

{}
scratch/broken.json is not valid JSON (line 1, column 17)


In [9]:
# Raise your own errors to reject bad data early, with a useful message.
class InvalidContactError(ValueError):
    """Raised when a contact record fails validation."""
    # A tiny class that "is a" ValueError. Classes come on Day 5.

def check_email(email: str) -> str:
    """Return the cleaned email, or raise InvalidContactError."""
    cleaned = email.strip().lower()
    if "@" not in cleaned:
        # raise = JS "throw": stop here and hand the error to the caller
        raise InvalidContactError(f"email {email!r} has no '@'")
    return cleaned

print(check_email("  Ravi@Globex.com "))  # -> ravi@globex.com
for value in ["meera@initech.io", "not-an-email"]:
    try:
        print("ok:", check_email(value))
    except InvalidContactError as err:  # catches only OUR error type
        print("rejected:", err)
# -> ok: meera@initech.io
# -> rejected: email 'not-an-email' has no '@'

ravi@globex.com
ok: meera@initech.io
rejected: email 'not-an-email' has no '@'


**Remember:** catch specific exceptions where you can do something useful, keep `try` blocks small, never `except: pass`. Use `raise NewError(...) from err` to add context without losing the cause. EAFP (try it, handle failure) is the Python style.

## Concept 4: Modules and imports

Any `.py` file is a module. `import x` runs the file **once** and caches it. Code under `if __name__ == "__main__":` runs only when you execute the file directly, never on import.

The first cell writes `text_utils.py` next to this notebook so the second cell can import it (in a real project you would just create the file).

In [10]:
# Create text_utils.py next to the notebook (normally you write this file by hand).
from pathlib import Path

module_source = '# text_utils.py: a module is just a .py file whose names other files import.\n"""Small helpers for cleaning text fields from CMS and CRM exports."""\n\nDEFAULT_LOCALE = "en-in"  # a module-level constant (UPPER_CASE by convention)\n\ndef slugify(title: str) -> str:\n    """Turn \'Pricing Page 2026\' into \'pricing-page-2026\' for a URL."""\n    return "-".join(title.lower().split())\n\nprint("text_utils loaded, __name__ =", __name__)  # runs ONCE, on first import\n\nif __name__ == "__main__":\n    # True only when you run "python text_utils.py" directly, never on import.\n    # Like a file that exports functions AND has its own little demo.\n    print(slugify("Quick Self Test"))  # -> quick-self-test\n'
Path("text_utils.py").write_text(module_source, encoding="utf-8")
print(Path("text_utils.py").exists())  # -> True

True


In [11]:
# main.py: three ways to import, all from the same folder.
# whole module: call text_utils.slugify(...), like  import * as tu  in JS
import text_utils
# pick names: like  import { slugify } from "./text_utils"  in JS
from text_utils import slugify, DEFAULT_LOCALE
import text_utils as tu  # an alias; the module is NOT loaded again (cached)

print(text_utils.slugify("Pricing Page 2026"))  # -> pricing-page-2026
print(slugify("About Us"), DEFAULT_LOCALE)  # -> about-us en-in
print(tu is text_utils)  # -> True  (same cached module object)

text_utils loaded, __name__ = text_utils
pricing-page-2026
about-us en-in
True


In [12]:
# A few standard-library modules you will use constantly (batteries included).
import json  # JSON text <-> Python
from datetime import date, datetime, timezone  # dates and times
from pathlib import Path  # files and folders
import os  # operating-system tools, e.g. environment variables

print(date(2026, 10, 28).isoformat())  # -> 2026-10-28  (ISO: safe for JSON)
stamp = datetime(2026, 10, 28, 8, 0, tzinfo=timezone.utc)
print(json.dumps({"saved_at": stamp.isoformat()}))
# -> {"saved_at": "2026-10-28T08:00:00+00:00"}
print(Path("day04/contacts.json").suffix)  # -> .json
# FDE TIP: secrets come from environment variables, never from code
print(os.environ.get("CLAUDE_API_KEY_DEMO", "not set"))  # -> not set

2026-10-28
{"saved_at": "2026-10-28T08:00:00+00:00"}
.json
not set


**Remember:** never name your own file `json.py`, `csv.py` or `random.py` (it shadows the standard library). Keep prints and real work under the `__main__` guard.

## Concept 5: Putting it together, a safe JSON store

Write to a temporary file in the same folder, then `os.replace()` it over the real file. Readers see the complete old file or the complete new one, never half: an **atomic write**.

In [13]:
# Safe save: write to a temporary file first, then swap it in one step.
import json
import os
from pathlib import Path

def save_json(path: Path, data) -> None:
    """Write JSON so a crash mid-write never leaves a half-written file."""
    # parents=True also creates any missing parent folders
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")  # settings.json.tmp, same folder
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    # swap in one step: readers see the old file or the new one, never half
    os.replace(tmp, path)

target = Path("scratch") / "app" / "settings.json"
save_json(target, {"theme": "dark", "page_size": 20})
save_json(target, {"theme": "light", "page_size": 50})  # overwrite safely
data = json.loads(target.read_text(encoding="utf-8"))
print(data)  # -> {'theme': 'light', 'page_size': 50}
print(sorted(p.name for p in target.parent.iterdir()))  # -> ['settings.json']
# (no .tmp file is left behind)

{'theme': 'light', 'page_size': 50}
['settings.json']


**Remember:** temp file + `os.replace` in the same folder, and put a `"version"` field in every file format you invent.

## Check your understanding (do NOT run until you have predicted)

Write your predictions here first, then paste the code into a new cell and run it.

```python
# Predict each output BEFORE you run this cell.
import json

# 1
print(json.dumps({"ok": True, "n": None}))

# 2
print(json.loads(json.dumps({2: (1, 2)})))

# 3
try:
    print(int("12") + int("x"))
except ValueError:
    print("bad number")
else:
    print("no error")
finally:
    print("done")

# 4
def risky():
    try:
        return "from try"
    finally:
        print("cleanup")

print(risky())

# 5
try:
    {"a": 1}["b"]
except (KeyError, IndexError) as err:
    print(type(err).__name__, err)
```

*Your predictions:*



## Today's task

Open `contact_store.py`, `main.py` and `dsa.py` in this folder. Each has the brief, acceptance criteria and step-by-step hints. When done, paste your work into the Forward Deployment Engineer project chat as **"Day 4 review"**.